# DeepGlobe Road Extraction — Exploratory Data Analysis

This notebook is the **first step** in our road segmentation pipeline. Before writing a single line of model code, we need to deeply understand the data. Every downstream decision — architecture, loss function, augmentations, preprocessing, train/val split — should be justified by what we find here.

### What we're looking for

| Question | Why it matters |
|----------|---------------|
| How many samples per split? Do all splits have masks? | Determines our train/val strategy |
| Are all images the same size and mode? | Determines if we need resize/conversion in preprocessing |
| Are masks truly binary? What format? | Determines mask preprocessing (binarization threshold) |
| How imbalanced are road vs background pixels? | Drives loss function choice (Dice/Focal vs BCE) |
| Are there empty or near-empty masks? | May need to filter or handle specially |
| What do road patterns look like spatially? | Informs augmentation strategy |
| How diverse are the images? | Informs augmentation and generalization expectations |

In [ ]:
from pathlib import Path
import sys
import random
import warnings

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image

# Resolve project root whether running from notebooks/ or project root
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from road_segmentation.paths import DEEPGLOBE_DATASET_DIR
from road_segmentation.data.eda import (
    build_sample_table,
    compute_dataset_overview,
    discover_image_mask_pairs,
)

plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 100
pd.set_option("display.max_columns", 50)
warnings.filterwarnings("ignore")
random.seed(42)
np.random.seed(42)

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset path: {DEEPGLOBE_DATASET_DIR}")

---
## 1. Dataset Inventory

First question: **what do we actually have?** How many files per split, which splits have ground truth masks?

In [ ]:
dataset_root = DEEPGLOBE_DATASET_DIR
assert dataset_root.exists(), (
    f"Dataset not found at {dataset_root}.\n"
    "Run: python scripts/download_data.py"
)

# Count files per split
split_info = []
for split in ["train", "valid", "test"]:
    split_dir = dataset_root / split
    sat_files = sorted(split_dir.glob("*_sat.*"))
    mask_files = sorted(split_dir.glob("*_mask.*"))
    split_info.append({
        "split": split,
        "satellite_images": len(sat_files),
        "masks": len(mask_files),
        "has_ground_truth": len(mask_files) > 0,
    })

split_df = pd.DataFrame(split_info)
display(split_df)

print("\nKey observation: Only the train split has ground truth masks.")
print("We will need to create our own train/validation split from the train data.")

---
## 2. Image-Mask Pair Discovery & Integrity Check

Using the EDA utilities from `road_segmentation.data.eda`, we discover all valid image-mask pairs and build a metadata table. This checks:
- Whether every satellite image has a corresponding mask
- Image/mask dimensions match
- Mask values are binary
- Road pixel coverage per sample

In [ ]:
# Discover all image-mask pairs (scans train/ directory which has masks)
pairs = discover_image_mask_pairs(dataset_root)
print(f"Discovered {len(pairs):,} image-mask pairs")
print(f"\nFirst pair: {pairs[0]}")
print(f"Last pair:  {pairs[-1]}")

In [ ]:
# Build detailed metadata table for ALL samples
# This reads every image+mask — takes ~2-3 minutes for 6226 pairs
print("Building sample table (reading all images + masks)...")
sample_table = build_sample_table(pairs)
print(f"Table shape: {sample_table.shape}")
display(sample_table.head())

In [ ]:
# Aggregate statistics
overview = compute_dataset_overview(sample_table)
print("Dataset Overview")
print("=" * 50)
display(overview.to_frame("value"))

---
## 3. Image & Mask Properties

### 3a. Dimension and Mode Consistency
Are all images the same resolution? Same color mode? Any outliers?

In [ ]:
# Group by dimensions and modes
dimension_summary = (
    sample_table
    .groupby(["image_width", "image_height", "image_mode", "mask_mode"], dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)
print("Image/Mask dimension and mode combinations:")
display(dimension_summary)

# Size match check
mismatched = sample_table[~sample_table["size_matches"]]
print(f"\nSamples with mismatched image/mask dimensions: {len(mismatched)}")
if len(mismatched) > 0:
    display(mismatched[["sample_id", "image_width", "image_height", "mask_width", "mask_height"]].head(10))

### 3b. Mask Value Analysis

Critical question: Are masks stored as single-channel binary (0/1 or 0/255), or RGB? What preprocessing do we need?

In [ ]:
# Check mask format in detail
sample_mask_path = pairs[0].mask_path
with Image.open(sample_mask_path) as m:
    mask_arr = np.array(m)
    print(f"Mask file: {sample_mask_path.name}")
    print(f"Mask shape: {mask_arr.shape}")
    print(f"Mask dtype: {mask_arr.dtype}")
    print(f"Mask mode (PIL): {m.mode}")
    print(f"Value range: [{mask_arr.min()}, {mask_arr.max()}]")
    if mask_arr.ndim == 3:
        for c in range(mask_arr.shape[2]):
            print(f"  Channel {c} unique values: {np.unique(mask_arr[:,:,c])}")

print("\n--- Mask binary compliance across dataset ---")
print(f"Binary masks: {sample_table['mask_is_binary'].sum()} / {len(sample_table)}")
non_binary = sample_table[~sample_table["mask_is_binary"]]
if len(non_binary) > 0:
    print(f"Non-binary mask samples ({len(non_binary)}):")
    display(non_binary[["sample_id", "mask_unique_values"]].head(10))
else:
    print("All masks are binary ({0, 255} values only).")

print("\nImplication: Masks are RGB with road=white(255,255,255), bg=black(0,0,0).")
print("Preprocessing: Convert to single channel, threshold > 0 → binary {0, 1}.")

---
## 4. Class Imbalance Analysis

This is the **most important section** for training decisions. Road pixels are typically a tiny fraction of total pixels. The degree of imbalance directly determines our choice of loss function and evaluation metrics.

In [ ]:
coverage = sample_table["road_coverage_pct"]

print("Road Coverage Statistics (% of pixels that are road)")
print("=" * 50)
percentiles = coverage.quantile([0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
percentiles.index = [f"p{int(q*100)}" for q in percentiles.index]
stats = pd.concat([
    pd.Series({"mean": coverage.mean(), "std": coverage.std(), "min": coverage.min(), "max": coverage.max()}),
    percentiles
])
display(stats.to_frame("road_coverage_pct").round(3))

# Empty mask count
empty_count = (coverage == 0).sum()
near_empty = (coverage < 0.5).sum()
print(f"\nEmpty masks (0% coverage): {empty_count}")
print(f"Near-empty masks (<0.5% coverage): {near_empty}")
print(f"Dense masks (>15% coverage): {(coverage > 15).sum()}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Full distribution
axes[0].hist(coverage, bins=50, color="#2b8cbe", edgecolor="white", linewidth=0.5)
axes[0].axvline(coverage.mean(), color="red", linestyle="--", label=f"Mean: {coverage.mean():.1f}%")
axes[0].axvline(coverage.median(), color="orange", linestyle="--", label=f"Median: {coverage.median():.1f}%")
axes[0].set_title("Road Coverage Distribution")
axes[0].set_xlabel("Road pixels (%)")
axes[0].set_ylabel("Number of samples")
axes[0].legend()

# Zoomed: low coverage
low_cov = coverage[coverage < 5]
axes[1].hist(low_cov, bins=40, color="#f03b20", edgecolor="white", linewidth=0.5)
axes[1].set_title(f"Low Coverage Samples (<5%) — {len(low_cov)} samples")
axes[1].set_xlabel("Road pixels (%)")
axes[1].set_ylabel("Number of samples")

# Box plot
axes[2].boxplot(coverage, vert=True)
axes[2].set_title("Coverage Box Plot")
axes[2].set_ylabel("Road pixels (%)")

plt.tight_layout()
plt.show()

print(f"\nThe distribution is heavily right-skewed.")
print(f"{(coverage < 5).sum()}/{len(coverage)} samples ({(coverage < 5).mean()*100:.1f}%) have <5% road coverage.")
print(f"This confirms severe class imbalance — standard BCE loss will struggle.")

---
## 5. Spatial Analysis — Road Heatmap

By averaging all masks, we can see if roads have any spatial bias (e.g., more roads in the center). This informs whether center-crop augmentations might lose information.

In [ ]:
# Compute spatial heatmap from a random sample (for speed)
HEATMAP_SAMPLES = min(500, len(pairs))
heatmap_indices = random.sample(range(len(pairs)), HEATMAP_SAMPLES)

heatmap = np.zeros((1024, 1024), dtype=np.float64)
for idx in heatmap_indices:
    with Image.open(pairs[idx].mask_path) as m:
        mask_arr = np.array(m.convert("L"))  # Convert to grayscale
        heatmap += (mask_arr > 0).astype(np.float64)

heatmap /= HEATMAP_SAMPLES

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

im = axes[0].imshow(heatmap, cmap="hot", vmin=0)
axes[0].set_title(f"Road Frequency Heatmap (n={HEATMAP_SAMPLES})")
axes[0].set_xlabel("At each pixel: fraction of samples with a road")
plt.colorbar(im, ax=axes[0], label="Frequency")

# Row and column marginals
axes[1].plot(heatmap.mean(axis=0), label="Column mean", alpha=0.7)
axes[1].plot(heatmap.mean(axis=1), label="Row mean", alpha=0.7)
axes[1].set_title("Marginal Road Density")
axes[1].set_xlabel("Pixel position")
axes[1].set_ylabel("Mean road probability")
axes[1].legend()

plt.tight_layout()
plt.show()

print("If the heatmap is roughly uniform, roads don't have strong spatial bias.")
print("This means random crops and flips are safe augmentations — they won't systematically remove roads.")

---
## 6. Visual Inspection — Sample Grid

Looking at actual images is irreplaceable. We show a **4x4 grid** of random samples to understand the visual diversity: urban vs rural, different road widths, varying lighting conditions.

In [ ]:
def show_sample_grid(pair_list, title="Sample Grid", ncols=4, nrows=4):
    """Display a grid of image-mask overlay pairs."""
    fig, axes = plt.subplots(nrows, ncols * 2, figsize=(ncols * 5, nrows * 2.8))
    for i in range(nrows):
        for j in range(ncols):
            idx = i * ncols + j
            if idx >= len(pair_list):
                axes[i, j*2].axis("off")
                axes[i, j*2+1].axis("off")
                continue
            pair = pair_list[idx]
            cov = sample_table.loc[
                sample_table["sample_id"] == pair.sample_id, "road_coverage_pct"
            ].values[0]
            with Image.open(pair.image_path) as img:
                img_arr = np.array(img.convert("RGB"))
            with Image.open(pair.mask_path) as mask:
                mask_arr = np.array(mask.convert("L"))

            axes[i, j*2].imshow(img_arr)
            axes[i, j*2].set_title(f"{cov:.1f}%", fontsize=9)
            axes[i, j*2].axis("off")

            axes[i, j*2+1].imshow(mask_arr, cmap="gray")
            axes[i, j*2+1].axis("off")

    fig.suptitle(title, fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()


# Random sample grid
random_pairs = random.sample(pairs, 16)
show_sample_grid(random_pairs, title="Random Samples (image | mask)")

---
## 7. Edge Cases Gallery

Looking at extremes helps us anticipate training issues. We examine:
- **Sparsest** samples (lowest coverage) — are these valid? Annotation gaps?
- **Densest** samples (highest coverage) — highway interchanges? Urban cores?
- **Mid-range** samples — what "typical" looks like

In [ ]:
def show_overlay(pair, ax_img, ax_mask, ax_overlay, alpha=0.4):
    """Show image, mask, and overlay on provided axes."""
    with Image.open(pair.image_path) as img:
        img_arr = np.array(img.convert("RGB"))
    with Image.open(pair.mask_path) as mask:
        mask_arr = np.array(mask.convert("L"))

    overlay = img_arr.copy()
    road_px = mask_arr > 0
    overlay[road_px] = ((1 - alpha) * overlay[road_px] + alpha * np.array([255, 0, 0])).astype(np.uint8)

    cov = sample_table.loc[sample_table["sample_id"] == pair.sample_id, "road_coverage_pct"].values[0]

    ax_img.imshow(img_arr)
    ax_img.set_title(f"Image ({cov:.2f}% road)", fontsize=9)
    ax_img.axis("off")

    ax_mask.imshow(mask_arr, cmap="gray")
    ax_mask.set_title("Mask", fontsize=9)
    ax_mask.axis("off")

    ax_overlay.imshow(overlay)
    ax_overlay.set_title("Overlay", fontsize=9)
    ax_overlay.axis("off")


ranked = sample_table.sort_values("road_coverage_pct").reset_index(drop=True)

# Pick 3 sparsest, 3 densest, 3 mid-range
n = len(ranked)
categories = {
    "Sparsest (lowest road coverage)": ranked.head(3),
    "Mid-range (around median)": ranked.iloc[n//2 - 1 : n//2 + 2],
    "Densest (highest road coverage)": ranked.tail(3),
}

for cat_name, cat_rows in categories.items():
    fig, axes = plt.subplots(3, 3, figsize=(15, 15))
    fig.suptitle(cat_name, fontsize=14, fontweight="bold")
    for i, (_, row) in enumerate(cat_rows.iterrows()):
        pair = next(p for p in pairs if p.sample_id == row["sample_id"])
        show_overlay(pair, axes[i, 0], axes[i, 1], axes[i, 2])
    plt.tight_layout()
    plt.show()

---
## 8. Mask Quality — Edge Sharpness & Road Width

Understanding road width matters for choosing the right architecture (larger receptive field for wide roads, skip connections for thin ones). And mask edge quality tells us how much label noise to expect.

In [ ]:
# Estimate road width from a sample of masks using distance transform
import cv2  # noqa: E402 — imported here because it's only needed for this analysis

WIDTH_SAMPLES = min(200, len(pairs))
width_indices = random.sample(range(len(pairs)), WIDTH_SAMPLES)

all_widths = []
for idx in width_indices:
    with Image.open(pairs[idx].mask_path) as m:
        mask_arr = np.array(m.convert("L"))
    binary = (mask_arr > 0).astype(np.uint8)
    if binary.sum() == 0:
        continue
    # Distance transform gives distance from each road pixel to nearest edge
    dist = cv2.distanceTransform(binary, cv2.DIST_L2, 5)
    # Road width ≈ 2 × max distance along the skeleton
    # We take the mean of the top 10% distances as a robust width estimate
    road_distances = dist[dist > 0]
    if len(road_distances) > 0:
        p90_dist = np.percentile(road_distances, 90)
        all_widths.append(2 * p90_dist)  # diameter = 2 * radius

all_widths = np.array(all_widths)

fig, ax = plt.subplots(1, 1, figsize=(10, 4))
ax.hist(all_widths, bins=50, color="#7bccc4", edgecolor="white", linewidth=0.5)
ax.axvline(np.median(all_widths), color="red", linestyle="--", label=f"Median: {np.median(all_widths):.0f}px")
ax.set_title(f"Estimated Road Width Distribution (n={len(all_widths)})")
ax.set_xlabel("Estimated road width (pixels at 1024x1024)")
ax.set_ylabel("Count")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Road width stats (pixels at 1024x1024):")
print(f"  Min: {all_widths.min():.0f}, Median: {np.median(all_widths):.0f}, Max: {all_widths.max():.0f}")
print(f"  Mean: {all_widths.mean():.0f}, Std: {all_widths.std():.0f}")
print(f"\nIf we resize to 512x512, widths halve. Thin roads (~10px) become ~5px.")
print("This means we need an architecture with good skip connections (U-Net) to preserve thin features.")

---
## 9. Pixel Intensity Distribution

Understanding the RGB distribution of satellite images helps verify that ImageNet normalization is reasonable and check for systematic color biases.

In [ ]:
# Sample pixel statistics
PIXEL_SAMPLES = min(200, len(pairs))
pixel_indices = random.sample(range(len(pairs)), PIXEL_SAMPLES)

channel_means = []
channel_stds = []
for idx in pixel_indices:
    with Image.open(pairs[idx].image_path) as img:
        arr = np.array(img.convert("RGB")).astype(np.float32) / 255.0
    channel_means.append(arr.mean(axis=(0, 1)))
    channel_stds.append(arr.std(axis=(0, 1)))

channel_means = np.array(channel_means)
channel_stds = np.array(channel_stds)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ["red", "green", "blue"]
for c, color in enumerate(colors):
    axes[0].hist(channel_means[:, c], bins=30, alpha=0.6, color=color, label=f"{color.upper()} mean")
    axes[1].hist(channel_stds[:, c], bins=30, alpha=0.6, color=color, label=f"{color.upper()} std")

axes[0].set_title("Per-Image Channel Means")
axes[0].set_xlabel("Mean pixel value (0-1)")
axes[0].legend()

axes[1].set_title("Per-Image Channel Stds")
axes[1].set_xlabel("Std pixel value")
axes[1].legend()

plt.tight_layout()
plt.show()

dataset_mean = channel_means.mean(axis=0)
dataset_std = channel_stds.mean(axis=0)
print(f"Dataset channel means (RGB): [{dataset_mean[0]:.3f}, {dataset_mean[1]:.3f}, {dataset_mean[2]:.3f}]")
print(f"Dataset channel stds  (RGB): [{dataset_std[0]:.3f}, {dataset_std[1]:.3f}, {dataset_std[2]:.3f}]")
print(f"\nImageNet means: [0.485, 0.456, 0.406]")
print(f"ImageNet stds:  [0.229, 0.224, 0.225]")
print(f"\nUsing ImageNet normalization is standard for pretrained encoders and should work well here.")

---
## 10. Problematic Samples

Flag any samples that might cause issues during training.

In [ ]:
# Identify problematic samples
issues = []

# Size mismatches
size_mismatch = sample_table[~sample_table["size_matches"]]
if len(size_mismatch) > 0:
    issues.append(f"Size mismatches: {len(size_mismatch)} samples")
    display(size_mismatch[["sample_id", "image_width", "image_height", "mask_width", "mask_height"]])

# Non-binary masks
non_binary = sample_table[~sample_table["mask_is_binary"]]
if len(non_binary) > 0:
    issues.append(f"Non-binary masks: {len(non_binary)} samples")
    display(non_binary[["sample_id", "mask_unique_values"]])

# Empty masks
empty_masks = sample_table[sample_table["road_coverage_pct"] == 0]
if len(empty_masks) > 0:
    issues.append(f"Empty masks: {len(empty_masks)} samples")
    display(empty_masks[["sample_id", "road_coverage_pct"]])

# Near-empty masks (<0.1%)
near_empty = sample_table[sample_table["road_coverage_pct"] < 0.1]
if len(near_empty) > 0:
    issues.append(f"Near-empty masks (<0.1% coverage): {len(near_empty)} samples")

if issues:
    print("Issues found:")
    for issue in issues:
        print(f"  - {issue}")
else:
    print("No structural issues found! All samples have matching sizes, binary masks, and non-empty coverage.")

---
## 11. Decisions from EDA

Every observation above maps to a concrete engineering decision. This section connects the dots.

In [ ]:
mean_cov = coverage.mean()
median_cov = coverage.median()
all_binary = sample_table["mask_is_binary"].all()
all_same_size = (sample_table["image_width"] == 1024).all() and (sample_table["image_height"] == 1024).all()
masks_are_rgb = (sample_table["mask_mode"] == "RGB").all()

decisions = [
    {
        "topic": "Preprocessing — Mask Conversion",
        "observation": f"Masks are {'RGB' if masks_are_rgb else 'grayscale'} with values in {{0, 255}}. All masks are binary: {all_binary}.",
        "decision": "Convert masks to single-channel grayscale, then binarize: pixel > 0 → 1, else 0. No complex thresholding needed.",
    },
    {
        "topic": "Preprocessing — Resize",
        "observation": f"All images are {'1024x1024' if all_same_size else 'variable size'}. Consistent dimensions simplify the pipeline.",
        "decision": "Resize to 512x512 for training (4x fewer pixels → faster training). Road widths will halve but remain detectable. Can use 1024x1024 at inference for better quality.",
    },
    {
        "topic": "Loss Function",
        "observation": f"Mean road coverage: {mean_cov:.1f}%, median: {median_cov:.1f}%. Severe class imbalance — background dominates.",
        "decision": "Use BCE + Dice combined loss. BCE provides stable gradients; Dice directly optimizes the overlap metric and handles imbalance. Focal loss is an alternative but Dice is simpler and proven for road segmentation.",
    },
    {
        "topic": "Evaluation Metrics",
        "observation": f"With ~{mean_cov:.0f}% road coverage, a model predicting all-background gets ~{100-mean_cov:.0f}% pixel accuracy.",
        "decision": "Primary metric: IoU (Intersection over Union) for road class. Secondary: Dice coefficient, Precision, Recall. Pixel accuracy is NOT useful due to imbalance.",
    },
    {
        "topic": "Train/Val Split",
        "observation": f"Only train split ({len(pairs)} samples) has ground truth. Valid/test have no masks.",
        "decision": "Split train 85/15 into train/val. Stratify by road_coverage_pct to ensure both splits have similar coverage distributions.",
    },
    {
        "topic": "Augmentations",
        "observation": "Road heatmap shows roughly uniform spatial distribution. Images show diverse geography and lighting.",
        "decision": "Use: HorizontalFlip, VerticalFlip, RandomRotate90, RandomBrightnessContrast, HueSaturationValue. These are safe given uniform spatial distribution and add lighting robustness.",
    },
    {
        "topic": "Architecture",
        "observation": "Road widths vary from thin (~10px) to wide (~60+px). Thin roads need fine-grained features.",
        "decision": "U-Net with pretrained ResNet34 encoder. U-Net's skip connections preserve fine spatial detail for thin roads. ResNet34 is a good speed/accuracy tradeoff. Pretrained on ImageNet for faster convergence.",
    },
    {
        "topic": "Normalization",
        "observation": f"Dataset RGB means: [{dataset_mean[0]:.3f}, {dataset_mean[1]:.3f}, {dataset_mean[2]:.3f}]",
        "decision": "Use ImageNet normalization (mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) since we're using a pretrained encoder.",
    },
]

print("EDA-Driven Decisions")
print("=" * 70)
for d in decisions:
    print(f"\n{d['topic']}")
    print(f"  Observation: {d['observation']}")
    print(f"  Decision:    {d['decision']}")

---
## Summary

### Dataset at a Glance
- **6,226 training pairs** (satellite image + binary road mask), all 1024x1024 RGB
- **No ground truth** for validation/test splits — we create our own split
- **Severe class imbalance**: mean ~4% road coverage, median ~2%
- **Clean masks**: all binary, all matching dimensions, no corruptions

### Key Training Implications
1. **Loss**: BCE + Dice (not plain BCE — imbalance makes it ineffective)
2. **Metrics**: IoU primary, Dice secondary (pixel accuracy is meaningless)
3. **Architecture**: U-Net + ResNet34 encoder (skip connections for thin roads)
4. **Input**: 512x512 resize, ImageNet normalization
5. **Masks**: RGB → grayscale → binary (threshold > 0)
6. **Split**: 85/15 stratified by coverage
7. **Augmentations**: Flips, rotations, color jitter (safe given uniform spatial distribution)